# Iframe Cookie + X-Frame-Options Fix Verification — obsidian-tc

**Using pygraphistry 0.45.9** to avoid the `active_organization` requirement in 0.50.x.

**Fixes applied to `obsidian-tc.grph.xyz`:**
1. Caddyfile: `X-Frame-Options SAMEORIGIN` → `ALLOWALL` (allows cross-origin iframes)
2. `custom.env`: `COOKIE_SECURE=True` → `SameSite=None; Secure` on all cookies (allows cross-origin cookie sending)
3. User `default_organization` set to personal org (not SITE) to enable dataset creation

**Expected results after fix:**
| Test | Privacy | Expected |
|------|---------|----------|
| A | None (server default) | **PASS** — iframe renders viz (cookies sent cross-origin) |
| B | `mode='public'` | **PASS** — no auth needed |
| C | `mode='private'` | **PASS** — iframe renders viz (cookies sent cross-origin) |

In [ ]:
# Use 0.45.9 to avoid active_organization requirement
%pip install graphistry==0.45.9 requests
dbutils.library.restartPython()

In [ ]:
import graphistry
import requests
import pandas as pd

# --- Configuration ---
ACTIVE_SERVER = "obsidian-tc.grph.xyz"
PROTOCOL = "https"

print(f"graphistry version: {graphistry.__version__}")
print(f"Target server:      {PROTOCOL}://{ACTIVE_SERVER}")

In [ ]:
# SSO login (site-wide SSO - no org_name since SSO is configured at site level)
graphistry.register(
    api=3,
    protocol=PROTOCOL,
    server=ACTIVE_SERVER,
    is_sso_login=True,
    sso_opt_into_type="display",
    sso_timeout=None,
)
print("Click the SSO link above, complete login, then run the next cell.")

In [ ]:
# Retrieve SSO token
token = graphistry.sso_get_token()
assert token is not None and len(token) > 10, f"Token missing or invalid: {repr(token[:20] if token else None)}"
print(f"Token obtained: {token[:20]}...")
print(f"Server: {ACTIVE_SERVER}")
print(f"Version: {graphistry.__version__}")

# Verify token against server
try:
    is_valid = graphistry.verify_token()
    print(f"Token valid: {is_valid}")
except Exception as e:
    print(f"Token verify error: {e}")

In [ ]:
# Test data
edges = pd.DataFrame({
    "src": ["a", "b", "c", "d", "e"],
    "dst": ["b", "c", "d", "e", "a"],
    "weight": [1, 2, 3, 4, 5],
})

g = graphistry.edges(edges, "src", "dst")
print(f"Graph: {len(edges)} edges")
print(f"g._privacy = {getattr(g, '_privacy', 'MISSING')}")
print(f"session.privacy = {getattr(graphistry.PyGraphistry._config, 'privacy', 'MISSING')}")

In [ ]:
# === Test Matrix: upload 3 datasets with different privacy settings ===
# Use render=False to get URLs without rendering iframes

results = {}

# --- Test A: No privacy set (server default) ---
print("=" * 60)
print("TEST A: No privacy set (server default)")
url_a = g.plot(render=False)
print(f"  URL: {url_a}")
results['A_no_privacy'] = {'url': url_a}

# --- Test B: privacy(mode='public') ---
print("\n" + "=" * 60)
print("TEST B: privacy(mode='public')")
url_b = g.privacy(mode='public').plot(render=False)
print(f"  URL: {url_b}")
results['B_public'] = {'url': url_b}

# --- Test C: privacy(mode='private') ---
print("\n" + "=" * 60)
print("TEST C: privacy(mode='private')")
url_c = g.privacy(mode='private').plot(render=False)
print(f"  URL: {url_c}")
results['C_private'] = {'url': url_c}

# --- Check each URL for login redirect (unauthenticated GET) ---
print("\n" + "=" * 60)
print("Checking URLs without cookies (simulating iframe behavior)...")

for label, data in results.items():
    url = data['url']
    try:
        resp = requests.get(url, allow_redirects=True, timeout=15)
        final_url = resp.url
        is_login = any(x in final_url.lower() for x in ['login', 'signin', 'accounts/login'])
        has_login_form = 'login' in resp.text.lower()[:2000] if resp.text else False
        data['status'] = resp.status_code
        data['final_url'] = final_url
        data['redirected_to_login'] = is_login
        data['login_form_in_body'] = has_login_form
        data['shows_viz'] = not is_login and not has_login_form
        status = 'PASS (viz)' if data['shows_viz'] else 'FAIL (login page)'
        print(f"  {label}: {status}  [HTTP {resp.status_code}, redirect_to_login={is_login}]")
    except Exception as e:
        data['error'] = str(e)
        print(f"  {label}: ERROR - {e}")

In [ ]:
# === Inspect internal state ===
import inspect

print("=== Internal Privacy State ===")
print()

# Session-level privacy
session = graphistry.PyGraphistry._config
print(f"session.privacy: {getattr(session, 'privacy', 'MISSING')}")
print(f"session type:    {type(session).__name__}")
print()

# Graph-level privacy
g_plain = graphistry.edges(edges, "src", "dst")
g_pub = g_plain.privacy(mode='public')
g_priv = g_plain.privacy(mode='private')

print(f"g (no privacy)._privacy:  {g_plain._privacy}")
print(f"g.privacy('public')._privacy:  {g_pub._privacy}")
print(f"g.privacy('private')._privacy: {g_priv._privacy}")
print()

# Check maybe_post_share_link logic
from graphistry.arrow_uploader import ArrowUploader
print("maybe_post_share_link source:")
print(inspect.getsource(ArrowUploader.maybe_post_share_link))
print()

# Check cascade_privacy_settings defaults
if hasattr(ArrowUploader, 'cascade_privacy_settings'):
    src = inspect.getsource(ArrowUploader.cascade_privacy_settings)
    # Extract just the default lines
    for line in src.split('\n'):
        s = line.strip()
        if s.startswith('if mode is None') or s.startswith("mode = "):
            print(f"  cascade default: {s}")

In [ ]:
# === Render actual iframes for visual comparison ===
# In Databricks, displayHTML renders in the notebook output cell.

html_parts = []
html_parts.append("<h2>Privacy / Iframe Visual Comparison</h2>")
html_parts.append('<div style="display: flex; gap: 10px; flex-wrap: wrap;">')

for label, data in results.items():
    url = data.get('url', '')
    status = 'PASS' if data.get('shows_viz') else 'FAIL'
    color = '#2d7d2d' if status == 'PASS' else '#cc3333'
    html_parts.append(f'''
    <div style="border: 2px solid {color}; padding: 5px; min-width: 400px;">
        <h3 style="color: {color};">{label} ({status})</h3>
        <iframe src="{url}" width="400" height="350" style="border:1px solid #ccc;"></iframe>
        <p style="font-size:10px; word-break:break-all;">{url}</p>
    </div>
    ''')

html_parts.append('</div>')
html_out = '\n'.join(html_parts)

# Try Databricks displayHTML, fall back to IPython
try:
    displayHTML(html_out)
except NameError:
    from IPython.display import display, HTML
    display(HTML(html_out))

In [ ]:
# === Summary Table ===

print("=" * 80)
print(f"{'Test':<20} {'Privacy':<15} {'HTTP':<6} {'Login Redirect':<16} {'Login Form':<12} {'Result'}")
print("-" * 80)

for label, data in results.items():
    if 'error' in data:
        print(f"{label:<20} {'?':<15} {'ERR':<6} {'?':<16} {'?':<12} ERROR: {data['error'][:30]}")
    else:
        priv = 'none' if 'no_privacy' in label else ('public' if 'public' in label else 'private')
        result = 'PASS' if data.get('shows_viz') else 'FAIL'
        print(f"{label:<20} {priv:<15} {data.get('status','?'):<6} "
              f"{str(data.get('redirected_to_login','?')):<16} "
              f"{str(data.get('login_form_in_body','?')):<12} {result}")

print("=" * 80)
print()
print("Interpretation:")
print("  - If A (no privacy) FAILS but B (public) PASSES:")
print("    => Server default is 'private'; explicit public fixes it.")
print("  - If A and B both PASS but C (private) FAILS:")
print("    => Only explicit private causes the issue; server default is OK.")
print("  - If ALL PASS: iframe cookie fix is working!")
print()
print(f"Server tested: {ACTIVE_SERVER}")
print(f"graphistry version: {graphistry.__version__}")